# 🎬 Movie Recommender — Stage 1: Exploring the Data

Before building *any* model, a good data scientist **looks at the data first**.
This is called **Exploratory Data Analysis (EDA)**. We want to answer:

- What does each file contain?
- How are ratings distributed? (Do people mostly give high scores?)
- How active are users, and how popular are movies?
- How *sparse* is the data? (This shapes which models work.)

> **Before running this**, make sure you've downloaded the data — see the
> project README's "Get the data" section. The cells below expect
> `movies.csv` and `ratings.csv` in `../data/raw/`.

## 1. Setup

We import the libraries we'll use and tell pandas where the data lives.
`pandas` is the workhorse for tabular data; `matplotlib`/`seaborn` make charts.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")  # nicer default chart style

# Path to the raw data, relative to this notebook (notebooks/ -> ../data/raw)
RAW = Path("..") / "data" / "raw"
print("Looking for data in:", RAW.resolve())
print("Files found:", [p.name for p in RAW.glob("*.csv")])

## 2. Load the data

MovieLens splits its data across files. The two we care about most:

- **`movies.csv`** — one row per movie: `movieId`, `title`, `genres`.
- **`ratings.csv`** — one row per rating: `userId`, `movieId`, `rating`, `timestamp`.

The `ratings` table is the heart of a recommender: it records *who liked what*.

In [ ]:
movies = pd.read_csv(RAW / "movies.csv")
ratings = pd.read_csv(RAW / "ratings.csv")

print(f"movies:  {movies.shape[0]:,} rows x {movies.shape[1]} columns")
print(f"ratings: {ratings.shape[0]:,} rows x {ratings.shape[1]} columns")
print(f"\nUnique users:  {ratings['userId'].nunique():,}")
print(f"Unique movies rated: {ratings['movieId'].nunique():,}")

In [ ]:
# .head() shows the first few rows so we can eyeball the structure.
movies.head()

In [ ]:
ratings.head()

## 3. How are ratings distributed?

Ratings run from 0.5 to 5.0 stars. Knowing the distribution matters: if almost
everyone gives 4–5 stars, then "predicting a high rating" is easy and we need to
be careful about how we measure success later.

In [ ]:
ax = sns.countplot(data=ratings, x="rating", color="steelblue")
ax.set_title("Distribution of movie ratings")
ax.set_xlabel("Rating (stars)")
ax.set_ylabel("Number of ratings")
plt.show()

print("Average rating:", round(ratings["rating"].mean(), 2))

## 4. How active are users? How popular are movies?

Recommenders live or die by this. A handful of "super-users" rate thousands of
movies, while most users rate only a few. Likewise, a few blockbusters get tons
of ratings while most movies get almost none. This **long-tail** pattern is
everywhere in real-world data.

In [ ]:
ratings_per_user = ratings.groupby("userId").size()
ratings_per_movie = ratings.groupby("movieId").size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ratings_per_user, bins=50, color="steelblue")
axes[0].set_title("Ratings per user")
axes[0].set_xlabel("# ratings")
axes[0].set_ylabel("# users")

axes[1].hist(ratings_per_movie, bins=50, color="indianred")
axes[1].set_title("Ratings per movie")
axes[1].set_xlabel("# ratings")
axes[1].set_ylabel("# movies")

plt.tight_layout()
plt.show()

print(f"Median ratings per user:  {int(ratings_per_user.median())}")
print(f"Median ratings per movie: {int(ratings_per_movie.median())}")

## 5. What are the most-rated movies?

We join (`merge`) the rating counts back to the movie titles so the result is
human-readable. "Most rated" is our first hint at a **popularity baseline** —
the simplest possible recommender.

In [ ]:
movie_stats = (
    ratings.groupby("movieId")
    .agg(num_ratings=("rating", "size"), avg_rating=("rating", "mean"))
    .merge(movies[["movieId", "title"]], on="movieId")
)

top_rated = movie_stats.sort_values("num_ratings", ascending=False).head(10)
top_rated[["title", "num_ratings", "avg_rating"]].round(2)

## 6. How sparse is the data?

Imagine a giant grid: one row per user, one column per movie. Most cells are
**empty** because no one watches every movie. The fraction of filled cells is
the **density**; `1 - density` is the **sparsity**. Real recommender data is
typically >98% sparse — this is the central challenge the models must overcome.

In [ ]:
n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()
n_ratings = len(ratings)

density = n_ratings / (n_users * n_movies)
print(f"Users x Movies grid: {n_users:,} x {n_movies:,} = {n_users * n_movies:,} cells")
print(f"Actual ratings:      {n_ratings:,}")
print(f"Density:  {density:6.2%}  (fraction of cells filled)")
print(f"Sparsity: {1 - density:6.2%}  (fraction of cells empty)")

## What we learned & what's next

Jot down your takeaways here (recruiters love seeing your reasoning):

- Ratings skew toward **3–4 stars** — people tend to rate things they like.
- Both users and movies follow a **long tail** — a few are very active/popular.
- The data is extremely **sparse**, so our models must generalize from few signals.

**Next stage:** build a **popularity baseline** recommender, then improve on it
with content-based and collaborative-filtering models. Ask Claude to set up
`02_baseline.ipynb` when you're ready!